In [1]:
import pandas as pd
import matplotlib.pyplot as plt
# import seaborn as sns
import numpy as np

In [ ]:
data = pd.read_csv('C:\\Users\\Администратор\\Documents\\study\\APS\\df_model_aps_data.csv')
# data.columns.tolist()

['id',
 'referral_date',
 'open_date',
 'number_of_days_open',
 'disposition_status_description',
 'prior_aps_history_before_2022',
 'age_at_time_of_referral',
 'gender',
 'race',
 'marital_status_description',
 'veteran_ind',
 'housing_arrangement',
 'does_client_live_alone',
 'alleged_harm_self_neglect',
 'investigation_harm_self_neglect',
 'alleged_harm_neglect',
 'investigation_harm_neglect',
 'alleged_harm_financial_exploitation',
 'investigation_harm_financial_exploitation',
 'alleged_harm_exploitation_of_person',
 'investigation_harm_exploitation_of_person',
 'alleged_harm_physical_abuse',
 'investigation_harm_physical_abuse',
 'alleged_harm_emotional_abuse',
 'investigation_harm_emotional_abuse',
 'alleged_harm_sexual_abuse',
 'investigation_harm_sexual_abuse',
 'alleged_harm_abandonment',
 'investigation_harm_abandonment',
 'alleged_vulnerability_advanced_age_frailty_dependency',
 'alleged_vulnerability_cognitive_impairment',
 'alleged_vulnerability_developmental_intellectual_

In [8]:
data = data.drop(columns=data.columns[data.columns.str.startswith('reoccurred')], errors='ignore')
data.shape[1]

93

In [13]:
data = data.drop(columns=['time_since_first_open', 'main_id', 'first_open_date', 'duration'], errors='ignore')

In [14]:
# Make a copy of the original data to preserve it
df = data.copy()

# 1. Extract person-level ID
df['person_id'] = df['id'].str.split('-').str[0]

# Ensure date columns are datetime objects
df['open_date'] = pd.to_datetime(df['open_date'])
df['closed_date'] = pd.to_datetime(df['closed_date'])

# 2. Sort cases within each person by open_date
df = df.sort_values(by=['person_id', 'open_date']).reset_index(drop=True)

# 3. Create case_order
df['case_order'] = df.groupby('person_id').cumcount() + 1

# 4. Define t0 = first open_date for that person
df['t0'] = df.groupby('person_id')['open_date'].transform('min')

# 5 & 6. Create stop_time and event_next
# Get the next case's open_date
df['next_open_date'] = df.groupby('person_id')['open_date'].shift(-1)

# event_next: 1 if another APS case occurs, 0 otherwise
df['event_next'] = df['next_open_date'].notna().astype(int)

# Define end of observation period as 3 years after t0
df['end_of_obs'] = df['t0'] + pd.DateOffset(years=3)

# Calculate stop_time (in days)
df['stop_time'] = np.where(
    df['event_next'] == 1,
    (df['next_open_date'] - df['t0']).dt.days,
    (df['end_of_obs'] - df['t0']).dt.days
)

# 7. Keep required rows and original case-level predictors
# cols_to_keep = [
#     'person_id', 'case_order', 'stop_time', 'event_next',
#     'id', 'open_date', 'closed_date', 'number_of_days_open'
# ]
survival_df = df.copy()

# 8. Output the final dataset sorted by person ID and case_order
survival_df = survival_df.sort_values(['person_id', 'case_order']).reset_index(drop=True)

survival_df.head(10)

,id,referral_date,open_date,number_of_days_open,disposition_status_description,prior_aps_history_before_2022,age_at_time_of_referral,gender,race,marital_status_description,...,perpetrator_intial_risk_mental_emotional_health_control_over_client,total_case_count,closed_date,person_id,case_order,t0,next_open_date,event_next,end_of_obs,stop_time
0,987679398-2,2024-09-04,2024-09-04,15.0,CLOSED,Y,80.0,M,Caucasian/White,Single-Never Married,...,1,2,2024-09-19,987679398,1,2024-09-04,NaT,0,2027-09-04,1095.0
1,987679482-1,2022-03-01,2022-03-01,45.0,CLOSED,N,57.0,M,Caucasian/White,Married-Living Together,...,2,10,2022-04-15,987679482,1,2022-03-01,2023-02-13,1,2025-03-01,349.0
2,987679482-3,2023-02-13,2023-02-13,59.0,CLOSED,N,58.0,M,Caucasian/White,Single-Never Married,...,1,10,2023-04-13,987679482,2,2022-03-01,2024-10-09,1,2025-03-01,953.0
3,987679482-7,2024-10-09,2024-10-09,29.0,CLOSED,N,NaN,M,Caucasian/White,Married-Living Together,...,1,10,2024-11-07,987679482,3,2022-03-01,NaT,0,2025-03-01,1096.0
4,987679506-1,2024-10-31,2024-10-31,50.0,CLOSED,N,56.0,F,Caucasian/White,Married-Living Together,...,2,1,2024-12-20,987679506,1,2024-10-31,NaT,0,2027-10-31,1095.0
5,987679558-1,2023-06-08,2023-06-08,13.0,CLOSED,N,69.0,M,Caucasian/White,NaN,...,1,1,2023-06-21,987679558,1,2023-06-08,NaT,0,2026-06-08,1096.0
6,987679646-1,2022-12-14,2022-12-14,40.0,CLOSED,Y,58.0,F,Caucasian/White,Single-Never Married,...,1,2,2023-01-23,987679646,1,2022-12-14,2023-03-31,1,2025-12-14,107.0
7,987679646-2,2023-03-31,2023-03-31,27.0,CLOSED,Y,58.0,F,Unable to determine,Single-Never Married,...,1,2,2023-04-27,987679646,2,2022-12-14,NaT,0,2025-12-14,1096.0
8,987679670-1,2022-01-11,2022-01-11,30.0,CLOSED,Y,81.0,F,Caucasian/White,Widowed,...,1,2,2022-02-10,987679670,1,2022-01-11,2022-02-16,1,2025-01-11,36.0
9,987679670-2,2022-02-16,2022-02-16,26.0,CLOSED,Y,81.0,F,Caucasian/White,Widowed,...,2,2,2022-03-14,987679670,2,2022-01-11,NaT,0,2025-01-11,1096.0


In [20]:
survival_df.isna().sum().sort_values(ascending=False).head(15)

next_open_date                                                  52441
does_client_live_alone                                          19005
marital_status_description                                      17129
race                                                             3849
age_at_time_of_referral                                          1628
perpetrator_type_desc                                             398
primary_caregiver_ind                                             398
live_with_ind                                                     398
verified_vulnerability_cognitive_impairment                        29
verified_vulnerability_advanced_age_frailty_dependency             29
verified_vulnerability_developmental_intellectual_disability       29
number_of_days_open                                                29
verified_vulnerability_medically_fragile                           29
closed_date                                                        29
verified_vulnerabili

In [24]:
# dropping rows with small number of missing values
survival_df['does_client_live_alone'].value_counts()

does_client_live_alone
N    37651
Y    21293
Name: count, dtype: int64